<a href="https://colab.research.google.com/github/TaicirCheikhrouhou/IDSproject-QGAN-AE-RL-FL/blob/main/Federated_Learning_NSL_KDD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a id="1"></a>
# <div style="text-align:center; border-radius:25px 70px; padding:9px; color:white; margin:0; font-size:150%; font-family:Pacifico; background-color:#87CEEB; overflow:hidden"><b> Federated Learning Using NSL-KDD Dataset</b></div>

Federated Learning allows training machine learning models across multiple devices or organizations without sharing raw data.

**Benefits:**

* Privacy: Data stays local, protecting sensitive information.
* Decentralization: Train across multiple clients without centralizing data.
* Efficiency: Only model updates are shared, saving bandwidth.
* Personalization: Each client contributes to a global model while keeping local patterns.
* Scalability: Can handle thousands of clients, ideal for IoT or large networks.
* Collaboration: Enables organizations to build models together safely.

Applications: Healthcare, cybersecurity, finance, mobile apps, and IoT systems.

<a id="1"></a>
# <div style="text-align:center; border-radius:25px 70px; padding:9px; color:white; margin:0; font-size:150%; font-family:Pacifico; background-color:#87CEEB; overflow:hidden"><b>1. Imports & Configuration</b></div>

In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

<a id="1"></a>
# <div style="text-align:center; border-radius:25px 70px; padding:9px; color:white; margin:0; font-size:150%; font-family:Pacifico; background-color:#87CEEB; overflow:hidden"><b>2. Chargement NSL-KDD</b></div>

In [11]:
def load_nsl_kdd(train_path, test_path):
    columns = [
        "duration","protocol_type","service","flag","src_bytes","dst_bytes",
        "land","wrong_fragment","urgent","hot","num_failed_logins","logged_in",
        "num_compromised","root_shell","su_attempted","num_root",
        "num_file_creations","num_shells","num_access_files","num_outbound_cmds",
        "is_host_login","is_guest_login","count","srv_count","serror_rate",
        "srv_serror_rate","rerror_rate","srv_rerror_rate","same_srv_rate",
        "diff_srv_rate","srv_diff_host_rate","dst_host_count",
        "dst_host_srv_count","dst_host_same_srv_rate","dst_host_diff_srv_rate",
        "dst_host_same_src_port_rate","dst_host_srv_diff_host_rate",
        "dst_host_serror_rate","dst_host_srv_serror_rate",
        "dst_host_rerror_rate","dst_host_srv_rerror_rate","label","difficulty"
    ]

    train_df = pd.read_csv(train_path, names=columns, header=None, encoding='ISO-8859-1')
    test_df  = pd.read_csv(test_path, names=columns, header=None, encoding='ISO-8859-1')

      # Supprimer 'difficulty' seulement si elle existe
    train_df.drop(columns=['difficulty'], inplace=True, errors='ignore')
    test_df.drop(columns=['difficulty'], inplace=True, errors='ignore')

    return train_df, test_df


In [12]:
train_url = "https://raw.githubusercontent.com/defcom17/NSL_KDD/master/KDDTrain+.txt"
test_url  = "https://raw.githubusercontent.com/defcom17/NSL_KDD/master/KDDTest+.txt"

train_df, test_df = load_nsl_kdd(train_url, test_url)
print(train_df.shape, test_df.shape)
train_df.head()


(125973, 42) (22544, 42)


,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_srv_count,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,label
0,0,tcp,ftp_data,SF,491,0,0,0,0,0,...,25,0.17,0.03,0.17,0.00,0.00,0.00,0.05,0.00,normal
1,0,udp,other,SF,146,0,0,0,0,0,...,1,0.00,0.60,0.88,0.00,0.00,0.00,0.00,0.00,normal
2,0,tcp,private,S0,0,0,0,0,0,0,...,26,0.10,0.05,0.00,0.00,1.00,1.00,0.00,0.00,neptune
3,0,tcp,http,SF,232,8153,0,0,0,0,...,255,1.00,0.00,0.03,0.04,0.03,0.01,0.00,0.01,normal
4,0,tcp,http,SF,199,420,0,0,0,0,...,255,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,normal


<a id="1"></a>
#<div style="text-align:center; border-radius:25px 70px; padding:9px; color:white; margin:0; font-size:150%; font-family:Pacifico; background-color:#87CEEB; overflow:hidden"><b> 3. Prétraitement</b></div>

In [31]:
def preprocess_data(train_df, test_df):
    # Supprimer 'difficulty' si elle existe (avant de calculer numerical_cols)
    train_df.drop(columns=['difficulty'], inplace=True, errors='ignore')
    test_df.drop(columns=['difficulty'], inplace=True, errors='ignore')

    categorical_cols = ["protocol_type", "service", "flag"]
    numerical_cols = train_df.columns.drop(categorical_cols + ["label"], errors='ignore')  # safe

    # Labels binaires
    train_df["label"] = train_df["label"].apply(lambda x: 0 if x=="normal" else 1)
    test_df["label"]  = test_df["label"].apply(lambda x: 0 if x=="normal" else 1)

    # Encodage catégoriel
    encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
    X_train_cat = encoder.fit_transform(train_df[categorical_cols])
    X_test_cat  = encoder.transform(test_df[categorical_cols])


    # Normalisation
    scaler = StandardScaler()
    X_train_num = scaler.fit_transform(train_df[numerical_cols])
    X_test_num  = scaler.transform(test_df[numerical_cols])

    # Fusion
    X_train = np.hstack((X_train_num, X_train_cat))
    X_test  = np.hstack((X_test_num, X_test_cat))

    y_train = train_df["label"].values
    y_test  = test_df["label"].values

    return X_train, X_test, y_train, y_test


<a id="1"></a>
# <div style="text-align:center; border-radius:25px 70px; padding:9px; color:white; margin:0; font-size:150%; font-family:Pacifico; background-color:#87CEEB; overflow:hidden"><b> 4. Simulation des clients (Non-IID prêt)</b></div>

In [32]:
def split_clients(X, y, num_clients):
    clients = []
    X_split = np.array_split(X, num_clients)
    y_split = np.array_split(y, num_clients)

    for i in range(num_clients):
        clients.append((X_split[i], y_split[i]))

    return clients

<a id="1"></a>
# <div style="text-align:center; border-radius:25px 70px; padding:9px; color:white; margin:0; font-size:150%; font-family:Pacifico; background-color:#87CEEB; overflow:hidden"><b> 5. Modèle IDS (MLP)</b></div>

In [33]:
def create_model(input_dim):
    model = tf.keras.Sequential([
        tf.keras.layers.Dense(128, activation="relu", input_shape=(input_dim,)),
        tf.keras.layers.Dense(64, activation="relu"),
        tf.keras.layers.Dense(1, activation="sigmoid")
    ])

    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model

<a id="1"></a>
# <div style="text-align:center; border-radius:25px 70px; padding:9px; color:white; margin:0; font-size:150%; font-family:Pacifico; background-color:#87CEEB; overflow:hidden"><b>6. Federated Learning</b></div>

In [34]:
def federated_training(global_model, clients, X_test, y_test, rounds=5):
    num_clients = len(clients)

    for r in range(rounds):
        print(f"\n🔁 Round {r+1}/{rounds}")
        local_weights = []

        for i, (X_client, y_client) in enumerate(clients):
            print(f"  🧑 Client {i+1} training...")

            local_model = create_model(X_client.shape[1])
            local_model.set_weights(global_model.get_weights())

            local_model.fit(
                X_client, y_client,
                epochs=1,
                batch_size=32,
                verbose=0
            )

            local_weights.append(local_model.get_weights())

        # Federated Averaging
        new_weights = []
        for weights in zip(*local_weights):
            new_weights.append(np.mean(weights, axis=0))

        global_model.set_weights(new_weights)

        loss, acc = global_model.evaluate(X_test, y_test, verbose=0)
        print(f"  🌍 Global Accuracy: {acc:.4f}")

    return global_model

<a id="1"></a>
# <div style="text-align:center; border-radius:25px 70px; padding:9px; color:white; margin:0; font-size:150%; font-family:Pacifico; background-color:#87CEEB; overflow:hidden"><b>7. Évaluation IDS</b></div>


In [35]:
def evaluate_model(model, X_test, y_test):
    y_pred = (model.predict(X_test) > 0.5).astype(int)

    print("\n📊 Classification Report:")
    print(classification_report(y_test, y_pred))

    print("📉 Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))


<a id="1"></a>
# <div style="text-align:center; border-radius:25px 70px; padding:9px; color:white; margin:0; font-size:150%; font-family:Pacifico; background-color:#87CEEB; overflow:hidden"><b> 8. MAIN</b></div>

In [36]:
if __name__ == "__main__":
    # URLs directes pour Colab
    train_url = "https://raw.githubusercontent.com/defcom17/NSL_KDD/master/KDDTrain+.txt"
    test_url  = "https://raw.githubusercontent.com/defcom17/NSL_KDD/master/KDDTest+.txt"

    #  Charger les datasets depuis GitHub
    train_df, test_df = load_nsl_kdd(train_url, test_url)

    # Prétraitement (encodage, normalisation, labels)
    X_train, X_test, y_train, y_test = preprocess_data(train_df, test_df)

    # Simulation des clients pour Federated Learning
    num_clients = 3
    clients = split_clients(X_train, y_train, num_clients)

    # Créer le modèle global MLP
    global_model = create_model(X_train.shape[1])

    # Boucle de Federated Learning (MANUEL)
    global_model = federated_training(
        global_model,
        clients,
        X_test,
        y_test,
        rounds=5
    )

    # Évaluation finale sur le test set
    evaluate_model(global_model, X_test, y_test)



🔁 Round 1/5
  🧑 Client 1 training...
  🧑 Client 2 training...
  🧑 Client 3 training...
  🌍 Global Accuracy: 0.7723

🔁 Round 2/5
  🧑 Client 1 training...
  🧑 Client 2 training...
  🧑 Client 3 training...
  🌍 Global Accuracy: 0.7838

🔁 Round 3/5
  🧑 Client 1 training...
  🧑 Client 2 training...
  🧑 Client 3 training...
  🌍 Global Accuracy: 0.7883

🔁 Round 4/5
  🧑 Client 1 training...
  🧑 Client 2 training...
  🧑 Client 3 training...
  🌍 Global Accuracy: 0.7883

🔁 Round 5/5
  🧑 Client 1 training...
  🧑 Client 2 training...
  🧑 Client 3 training...
  🌍 Global Accuracy: 0.7947
705/705 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step

📊 Classification Report:
              precision    recall  f1-score   support

           0       0.69      0.95      0.80      9711
           1       0.95      0.68      0.79     12833

    accuracy                           0.79     22544
   macro avg       0.82      0.81      0.79     22544
weighted avg       0.84      0.79      0.79     22544

📉 Confusion Matrix:
[[9217